# DPDP Colab worker

This notebook connects Colab to the persistent coordinator at `dpdp.hari-pi.com`. It does **not** start ngrok or a public server inside Colab.

Before the first run, add `GITHUB_TOKEN` in Colab's **Secrets** panel (key icon), enable notebook access, select a GPU runtime, and choose **Runtime → Run all**. Keep the cell running while you want Colab to answer queued requests.

In [ ]:
import base64, pathlib, subprocess
from google.colab import userdata

repo = pathlib.Path('/content/DPDP-Benchmark')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add GITHUB_TOKEN in Colab Secrets and enable notebook access.')

auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_auth = f'http.extraHeader=Authorization: Basic {auth}'
if not (repo / '.git').is_dir():
    subprocess.run([
        'git', '-c', git_auth, 'clone',
        'https://github.com/Hari-Pi/DPDP-Benchmark.git', str(repo),
    ], check=True)
else:
    subprocess.run([
        'git', '-C', str(repo), '-c', git_auth, 'pull', '--ff-only',
    ], check=True)

print('Starting the outbound Colab worker. Leave this cell running.')
subprocess.run(['bash', 'scripts/start_colab_worker.sh'], cwd=repo, check=True)